In [1]:
"""
bc_detection_per_instance.py
============================
For every unique instance in BUMP_CSV, determines whether ANY model/variant
detected a breaking change. Outputs a CSV with a 'BC_detected?' column (Yes/No).
"""

import json
import csv
from pathlib import Path


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LLM_CONFIGS = [
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"),

    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),

    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),
]

BUMP_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"

OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/bc_detection_per_instance.csv"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


def parse_bump_errors(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        result.add(e.split('.')[-1])
    return '|'.join(sorted(result))


def load_bump(bump_csv: str) -> list[dict]:
    """Load all BUMP rows, preserving original columns."""
    rows = []
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            rows.append(row)
    return rows


def load_llm_results(llm_json: str) -> dict:
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    print("Loading BUMP data...")
    bump_rows = load_bump(BUMP_CSV)
    print(f"  BUMP instances: {len(bump_rows)}")

    # Collect all custom_ids that had BC detected in at least one config
    detected_anywhere = set()

    for model_name, context_variant, json_path in LLM_CONFIGS:
        print(f"Processing: {model_name} / {context_variant}")

        if not Path(json_path).is_file():
            print(f"  WARNING: File not found — skipping")
            continue

        llm = load_llm_results(json_path)

        for instance_id, instance_data in llm.items():
            tests_data = instance_data.get('tests', {})
            failed_tests = tests_data.get('failed', [])
            if len(failed_tests) > 0:
                detected_anywhere.add(instance_id)

    # Build output: one row per unique BUMP instance
    # Preserve original BUMP columns + add BC_detected?
    bump_fieldnames = None
    output_rows = []

    with open(BUMP_CSV, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        bump_fieldnames = list(reader.fieldnames)

        for row in reader:
            cid = row['custom_id']
            row['BC_detected?'] = 'Yes' if cid in detected_anywhere else 'No'
            output_rows.append(row)

    output_fieldnames = bump_fieldnames + ['BC_detected?']

    # Write output
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=output_fieldnames)
        writer.writeheader()
        writer.writerows(output_rows)

    # Summary stats
    total = len(output_rows)
    num_detected = sum(1 for r in output_rows if r['BC_detected?'] == 'Yes')
    num_undetected = total - num_detected

    print(f"\n{'='*60}")
    print(f"BC DETECTION PER UNIQUE BUMP INSTANCE")
    print(f"{'='*60}")
    print(f"Total BUMP instances:              {total}")
    print(f"BC detected (by any model/variant): {num_detected}")
    print(f"BC never detected (by any):         {num_undetected}")
    print(f"\nOutput: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Loading BUMP data...
  BUMP instances: 89
Processing: GPT-4o / Minimal
Processing: GPT-4o / Method
Processing: GPT-4o / Class
Processing: Qwen-480B / Minimal
Processing: Qwen-480B / Method
Processing: Qwen-480B / Class
Processing: GPTOSS-120b / Minimal
Processing: GPTOSS-120b / Method
Processing: GPTOSS-120b / Class

BC DETECTION PER UNIQUE BUMP INSTANCE
Total BUMP instances:              89
BC detected (by any model/variant): 32
BC never detected (by any):         57

Output: /Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/bc_detection_per_instance.csv


In [2]:
"""
bc_detection_per_instance.py
============================
For every unique instance in BUMP_CSV, determines whether ANY model/variant
detected a breaking change. Outputs a CSV with a 'BC_detected?' column (Yes/No).
"""

import json
import csv
from pathlib import Path


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LLM_CONFIGS = [
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"),

    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),

    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),
]

BUMP_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"

OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/bc_detection_per_instance.csv"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


def parse_bump_errors(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        result.add(e.split('.')[-1])
    return '|'.join(sorted(result))


def load_bump(bump_csv: str) -> list[dict]:
    """Load all BUMP rows, preserving original columns."""
    rows = []
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            rows.append(row)
    return rows


def load_llm_results(llm_json: str) -> dict:
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    print("Loading BUMP data...")
    bump_rows = load_bump(BUMP_CSV)
    print(f"  BUMP instances: {len(bump_rows)}")

    # Collect all custom_ids that had BC detected in at least one config
    detected_anywhere = set()

    for model_name, context_variant, json_path in LLM_CONFIGS:
        print(f"Processing: {model_name} / {context_variant}")

        if not Path(json_path).is_file():
            print(f"  WARNING: File not found — skipping")
            continue

        llm = load_llm_results(json_path)

        for instance_id, instance_data in llm.items():
            tests_data = instance_data.get('tests', {})
            failed_tests = tests_data.get('failed', [])
            if len(failed_tests) > 0:
                detected_anywhere.add(instance_id)

    # Build output: one row per unique BUMP instance
    # Preserve original BUMP columns + add BC_detected?
    bump_fieldnames = None
    output_rows = []

    with open(BUMP_CSV, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        bump_fieldnames = list(reader.fieldnames)

        for row in reader:
            cid = row['custom_id']
            row['BC_detected?'] = 'Yes' if cid in detected_anywhere else 'No'
            output_rows.append(row)

    output_fieldnames = bump_fieldnames + ['BC_detected?']

    # Write output
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=output_fieldnames)
        writer.writeheader()
        writer.writerows(output_rows)

    # Also track all unique instance IDs that appeared in ANY JSON file
    all_ids_in_jsons = set()
    for model_name, context_variant, json_path in LLM_CONFIGS:
        if not Path(json_path).is_file():
            continue
        llm = load_llm_results(json_path)
        all_ids_in_jsons.update(llm.keys())

    # Instances present in JSONs but never detected by any config
    in_json_never_detected = all_ids_in_jsons - detected_anywhere

    # Summary stats
    total = len(output_rows)
    num_detected = sum(1 for r in output_rows if r['BC_detected?'] == 'Yes')
    num_undetected = total - num_detected

    print(f"\n{'='*60}")
    print(f"BC DETECTION PER UNIQUE BUMP INSTANCE")
    print(f"{'='*60}")
    print(f"Total BUMP instances (from CSV):    {total}")
    print(f"BC detected (by any model/variant): {num_detected}")
    print(f"BC never detected (by any):         {num_undetected}")
    print(f"\n--- JSON-level stats ---")
    print(f"Unique instance IDs across all JSONs: {len(all_ids_in_jsons)}")
    print(f"Detected in at least one config:      {len(detected_anywhere)}")
    print(f"Present in JSONs but NEVER detected:  {len(in_json_never_detected)}")
    print(f"\nOutput: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Loading BUMP data...
  BUMP instances: 89
Processing: GPT-4o / Minimal
Processing: GPT-4o / Method
Processing: GPT-4o / Class
Processing: Qwen-480B / Minimal
Processing: Qwen-480B / Method
Processing: Qwen-480B / Class
Processing: GPTOSS-120b / Minimal
Processing: GPTOSS-120b / Method
Processing: GPTOSS-120b / Class

BC DETECTION PER UNIQUE BUMP INSTANCE
Total BUMP instances (from CSV):    89
BC detected (by any model/variant): 32
BC never detected (by any):         57

--- JSON-level stats ---
Unique instance IDs across all JSONs: 46
Detected in at least one config:      32
Present in JSONs but NEVER detected:  14

Output: /Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/bc_detection_per_instance.csv


"""
bc_detection_per_instance.py
============================
For every unique instance in BUMP_CSV, determines whether ANY model/variant
detected a breaking change. Outputs a CSV with a 'BC_detected?' column (Yes/No).
"""

import json
import csv
from pathlib import Path


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LLM_CONFIGS = [
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"),

    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),

    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),
]

# Compilation JSONs — same order as LLM_CONFIGS (model, variant, path)
COMPILE_CONFIGS = [
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/compile_results_pre.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/compile_results_pre.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json"),

    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json"),

    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json"),
]

BUMP_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"

OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/bc_detection_per_instance.csv"

In [7]:
"""
bc_detection_per_instance.py
============================
For every unique instance in BUMP_CSV, determines whether ANY model/variant
detected a breaking change. Outputs a CSV with a 'BC_detected?' column (Yes/No).
"""

import json
import csv
from pathlib import Path


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LLM_CONFIGS = [
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"),

    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),

    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),
]

# Compilation JSONs — same order as LLM_CONFIGS (model, variant, path)
COMPILE_CONFIGS = [
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/compile_results_pre.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/compile_results_pre.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json"),

    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json"),

    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json"),
]

BUMP_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"

OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/bc_detection_per_instance.csv"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


def parse_bump_errors(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        result.add(e.split('.')[-1])
    return '|'.join(sorted(result))


def load_llm_results(llm_json: str) -> dict:
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    # ── 1. Process compilation JSONs ──────────────────────────────────────────
    compiled_anywhere = set()
    all_ids_in_compile_jsons = set()

    for model_name, context_variant, json_path in COMPILE_CONFIGS:
        print(f"Processing compilation: {model_name} / {context_variant}")
        if not Path(json_path).is_file():
            print(f"  WARNING: File not found — skipping")
            continue

        with open(json_path) as f:
            compile_data = json.load(f)
        file_counts = compile_data.get('file_counts', compile_data)

        for instance_id, counts in file_counts.items():
            all_ids_in_compile_jsons.add(instance_id)
            if isinstance(counts, dict):
                files_compiled = counts.get('files_compiled', 0)
                if files_compiled > 0:
                    compiled_anywhere.add(instance_id)

    # ── 2. Process transplant/execution JSONs ─────────────────────────────────
    detected_anywhere = set()
    all_ids_in_jsons = set()

    for model_name, context_variant, json_path in LLM_CONFIGS:
        print(f"Processing execution: {model_name} / {context_variant}")
        if not Path(json_path).is_file():
            print(f"  WARNING: File not found — skipping")
            continue

        llm = load_llm_results(json_path)
        all_ids_in_jsons.update(llm.keys())

        for instance_id, instance_data in llm.items():
            tests_data = instance_data.get('tests', {})
            failed_tests = tests_data.get('failed', [])
            if len(failed_tests) > 0:
                detected_anywhere.add(instance_id)

    # ── 3. Build output CSV ───────────────────────────────────────────────────
    print("\nLoading BUMP CSV and writing output...")
    bump_fieldnames = None
    output_rows = []

    with open(BUMP_CSV, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        bump_fieldnames = list(reader.fieldnames)

        for row in reader:
            cid = row['custom_id']
            row['compiled?'] = 'Yes' if cid in compiled_anywhere else 'No'
            row['executed?'] = 'Yes' if cid in all_ids_in_jsons else 'No'
            row['BC_detected?'] = 'Yes' if cid in detected_anywhere else 'No'
            output_rows.append(row)

    output_fieldnames = bump_fieldnames + ['compiled?', 'executed?', 'BC_detected?']

    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=output_fieldnames)
        writer.writeheader()
        writer.writerows(output_rows)

    # ── 4. Console summary ────────────────────────────────────────────────────
    total = len(output_rows)
    num_compiled = sum(1 for r in output_rows if r['compiled?'] == 'Yes')
    num_executed = sum(1 for r in output_rows if r['executed?'] == 'Yes')
    num_detected = sum(1 for r in output_rows if r['BC_detected?'] == 'Yes')

    in_json_never_detected = all_ids_in_jsons - detected_anywhere

    print(f"\n{'='*60}")
    print(f"BC DETECTION PER UNIQUE BUMP INSTANCE")
    print(f"{'='*60}")
    print(f"Total BUMP instances (from CSV):    {total}")
    print(f"Compiled (any config):              {num_compiled}")
    print(f"Executed (any config):              {num_executed}")
    print(f"BC detected (any config):           {num_detected}")
    print(f"BC never detected (any config):     {total - num_detected}")

    print(f"\n--- JSON-level stats ---")
    print(f"Unique IDs across compile JSONs:      {len(all_ids_in_compile_jsons)}")
    print(f"Compiled in ≥1 config:                {len(compiled_anywhere)}")
    print(f"Never compiled in any config:          {len(all_ids_in_compile_jsons - compiled_anywhere)}")
    print(f"Unique IDs across execution JSONs:     {len(all_ids_in_jsons)}")
    print(f"Detected in ≥1 config:                {len(detected_anywhere)}")
    print(f"Present in JSONs but NEVER detected:  {len(in_json_never_detected)}")

    print(f"\nOutput: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Processing compilation: GPT-4o / Minimal
Processing compilation: GPT-4o / Method
Processing compilation: GPT-4o / Class
Processing compilation: Qwen-480B / Minimal
Processing compilation: Qwen-480B / Method
Processing compilation: Qwen-480B / Class
Processing compilation: GPTOSS-120b / Minimal
Processing compilation: GPTOSS-120b / Method
Processing compilation: GPTOSS-120b / Class
Processing execution: GPT-4o / Minimal
Processing execution: GPT-4o / Method
Processing execution: GPT-4o / Class
Processing execution: Qwen-480B / Minimal
Processing execution: Qwen-480B / Method
Processing execution: Qwen-480B / Class
Processing execution: GPTOSS-120b / Minimal
Processing execution: GPTOSS-120b / Method
Processing execution: GPTOSS-120b / Class

Loading BUMP CSV and writing output...

BC DETECTION PER UNIQUE BUMP INSTANCE
Total BUMP instances (from CSV):    89
Compiled (any config):              47
Executed (any config):              46
BC detected (any config):           32
BC never detect